In [1]:
from nemo.collections.asr.parts.submodules.wfst_decoder import RivaGpuWfstDecoder

from inference_funcs import load_bit_phoneme_model, evaluate_model
from dataset import getDatasetLoaders
import numpy as np
import torch
import torch.nn.functional as F

langugage_model_fst_path = "/data/code/nejm-brain-to-text/language_model/pretrained_language_models/openwebtext_1gram_lm_sil/TLG_with_symbols.fst"

decoder = RivaGpuWfstDecoder(lm_fst=langugage_model_fst_path, decoding_mode="nbest", 
                             beam_size=18, lm_weight=1.0, nbest_size=18, max_mem=400000000, blank_penalty=0.70)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = 'cuda'

bit_phoneme_filepath = "/data/models/transformer_short_training_fixed_seed_0/"
model, args = load_bit_phoneme_model(bit_phoneme_filepath)
model = model.to(device)

data_file = '/data/neural_data/ptDecoder_ctc_both'
trainLoaders, testLoaders, loadedData = getDatasetLoaders(
        data_file, 8, None, 
        False
    )

outputs, cer, per_day_cer = evaluate_model(model, loadedData, args, partition='test', device='cuda', verbose=False)

num_classes = 41
logits = np.zeros((len(outputs['logits']), max(outputs['logitLengths']), num_classes))
for idx, l in enumerate(outputs['logits']):
    l_length = outputs['logitLengths'][idx]
    logits[idx, :l_length, :] = l
    
logits_torch = torch.from_numpy(logits)
log_probs = F.log_softmax(logits_torch, dim=-1).to(dtype=torch.float32, device=device)
log_probs_length = torch.from_numpy(np.array(outputs['logitLengths'])).to(dtype=torch.int64, device='cpu')

In [3]:
log_probs_blank_last = torch.concat((log_probs[:, :, 1:], log_probs[:, :, 0:1]), dim=-1) # move blank to end
log_probs_arranged = torch.concat((log_probs_blank_last[:, :, -1:], log_probs_blank_last[:, :, -2:-1], log_probs_blank_last[:, :, :-2]), dim=-1)

In [4]:
hypotheses = decoder.decode(log_probs_arranged, log_probs_length)

LOG ([5.5]:RebuildRepository():lat/determinize-lattice-pruned.cc:287) Rebuilding repository.
LOG ([5.5]:RebuildRepository():lat/determinize-lattice-pruned.cc:287) Rebuilding repository.
LOG ([5.5]:RebuildRepository():lat/determinize-lattice-pruned.cc:287) Rebuilding repository.
LOG ([5.5]:RebuildRepository():lat/determinize-lattice-pruned.cc:287) Rebuilding repository.
LOG ([5.5]:RebuildRepository():lat/determinize-lattice-pruned.cc:287) Rebuilding repository.
LOG ([5.5]:RebuildRepository():lat/determinize-lattice-pruned.cc:287) Rebuilding repository.
LOG ([5.5]:RebuildRepository():lat/determinize-lattice-pruned.cc:287) Rebuilding repository.
WARNING ([5.5]:CheckMemoryUsage():lat/determinize-lattice-pruned.cc:320) Did not reach requested beam in determinize-lattice: size exceeds maximum 400000000 bytes; (repo,arcs,elems) = (261414848,3976064,138325848), after rebuilding, repo size was 197033472, effective beam was 17.1798 vs. requested beam 18
LOG ([5.5]:RebuildRepository():lat/determi

In [5]:
print(dir(hypotheses[0]))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_has_alignment', '_has_timesteps', '_hypotheses', '_shape0', '_shape1', 'has_alignment', 'has_timesteps', 'replace_unit_', 'shape0', 'shape1']


In [22]:
idx = 10
print(outputs['transcriptions'][idx])
print(hypotheses[idx]._hypotheses)

he talked about unauthentic storylines too
[WfstNbestUnit(words=('HE', 'CHALK', 'ABOUT', 'ON', 'ATHLETIC', 'CLINES', 'YOUR'), timesteps=(0, 21, 29, 40, 45, 66, 87), alignment=(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 17, 17, 19, 19, 0, 0, 1, 1, 1, 9, 9, 2, 0, 21, 21, 1, 1, 1, 4, 4, 8, 8, 6, 32, 32, 32, 1, 1, 0, 0, 2, 24, 24, 1, 1, 1, 0, 0, 3, 33, 0, 0, 22, 22, 22, 12, 32, 32, 32, 18, 21, 21, 21, 1, 1, 21, 21, 0, 0, 0, 0, 0, 0, 0, 0, 22, 0, 7, 24, 24, 39, 39, 1, 1, 1, 38, 38, 5, 29, 0, 1, 1, 1, 1, 1, 1, 1), score=115950.1171875), WfstNbestUnit(words=('HEE', 'CHALK', 'ABOUT', 'ON', 'ATHLETIC', 'CLINES', 'YOUR'), timesteps=(0, 21, 29, 40, 45, 66, 87), alignment=(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 17, 17, 19, 19, 0, 0, 1, 1, 1, 9, 9, 2, 0, 21, 21, 1, 1, 1, 4, 4, 8, 8, 6, 32, 32, 32, 1, 1, 0, 0, 2, 24, 24, 1, 1, 1, 0, 0, 3, 33, 0, 0, 22, 22, 22, 12, 32, 32, 32, 18, 21, 21, 21, 1, 1, 21, 21, 0, 0, 0, 0, 0, 0, 0, 0, 22, 0, 7, 24, 24, 39, 39, 1, 1, 1, 38, 38, 5, 29, 0, 1, 1, 1, 1, 1, 1, 1), score=

In [32]:
idx = 800
print(outputs['transcriptions'][idx])
for i in range(18):
    print(hypotheses[idx]._hypotheses[i])

they're welcome to go some place else
WfstNbestUnit(words=('THERE', 'WILLAM', 'TO', 'GO', 'SOME', 'MEISS', 'ELSE'), timesteps=(0, 29, 40, 44, 50, 57, 65), alignment=(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 11, 12, 29, 29, 1, 1, 1, 37, 18, 18, 22, 22, 22, 4, 4, 23, 1, 1, 32, 35, 35, 1, 16, 26, 26, 1, 1, 1, 30, 30, 4, 23, 23, 1, 1, 1, 23, 7, 7, 30, 1, 1, 1, 12, 12, 22, 22, 30, 30, 1, 1, 1, 1, 1, 1, 1), score=98770.46875)
WfstNbestUnit(words=('THEIR', 'WILLEM', 'TO', 'GO', 'SOME', 'MEISS', 'ELSE'), timesteps=(0, 29, 40, 44, 50, 57, 65), alignment=(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 11, 12, 29, 29, 1, 1, 1, 37, 18, 18, 22, 22, 22, 4, 4, 23, 1, 1, 32, 35, 35, 1, 16, 26, 26, 1, 1, 1, 30, 30, 4, 23, 23, 1, 1, 1, 23, 7, 7, 30, 1, 1, 1, 12, 12, 22, 22, 30, 30, 1, 1, 1, 1, 1, 1, 1), score=98770.46875)
WfstNbestUnit(words=('THEIR', 'WILLAM', 'TO', 'GAU', 'SUM', 'MICE', 'ELSE'), timesteps=(0, 29, 40, 44, 50, 57, 65), alignment=(0, 0, 0, 0, 0

In [36]:
# First second only (10 frames = 1s if each frame = 100ms)

log_probs_chunk = log_probs_arranged[:, 0:10, :]  # take first 10 frames
log_probs_len_chunk = torch.tensor([10] * log_probs_arranged.shape[0])  # length per batch

# Decode just this first-second chunk
hypotheses_first_second = decoder._decode_nbest(log_probs_chunk, log_probs_len_chunk)


log_probs_chunk = log_probs_arranged[:, 10:20, :]  # take first 10 frames
log_probs_len_chunk = torch.tensor([10] * log_probs_arranged.shape[0])  # length per batch

hypotheses_second_second = decoder._decode_nbest(log_probs_chunk, log_probs_len_chunk)

In [38]:
idx = 400
print(outputs['transcriptions'][idx])
for i in range(18):
    print(hypotheses_first_second[idx]._hypotheses[i])
    print(hypotheses_second_second[idx]._hypotheses[i])

friday afternoon at five thirty
WfstNbestUnit(words=('VOID',), timesteps=(0,), alignment=(0, 0, 0, 0, 0, 0, 36, 27, 27, 10), score=17819.146484375)
WfstNbestUnit(words=('DE', 'AFTER'), timesteps=(0, 4), alignment=(10, 19, 19, 1, 1, 3, 15, 15, 32, 13), score=33542.0625)
WfstNbestUnit(words=('VIDE',), timesteps=(0,), alignment=(0, 0, 0, 0, 0, 0, 36, 36, 7, 10), score=26646.03515625)
WfstNbestUnit(words=('DI', 'AFTER'), timesteps=(0, 4), alignment=(10, 19, 19, 1, 1, 3, 15, 15, 32, 13), score=33947.3359375)
WfstNbestUnit(words=('VIED',), timesteps=(0,), alignment=(0, 0, 0, 0, 0, 0, 36, 36, 7, 10), score=26646.03515625)
WfstNbestUnit(words=('DEE', 'AFTER'), timesteps=(0, 4), alignment=(10, 19, 19, 1, 1, 3, 15, 15, 32, 13), score=34640.6953125)
WfstNbestUnit(words=('VOIGHT',), timesteps=(0,), alignment=(0, 0, 0, 0, 0, 0, 36, 27, 27, 32), score=29343.560546875)
WfstNbestUnit(words=('DEA', 'AFTER'), timesteps=(0, 4), alignment=(10, 19, 19, 1, 1, 3, 15, 15, 32, 13), score=34640.6953125)
WfstNbe